# LLM Evaluation on Greek Protipa Exams

This notebook evaluates multiple LLMs on the `PennyK98/protipa_exams_dataset` from Hugging Face. It focuses on Greek Language and Mathematics multiple-choice questions.

In [8]:
import json
import logging
import os
import random
import time
import traceback
from pathlib import Path

import lm_eval
import pandas as pd
import yaml
from datasets import load_dataset
from dotenv import load_dotenv
from lm_eval.models.openai_completions import OpenAIChatCompletion
from lm_eval.tasks import ConfigurableTask, TaskManager

import IPython.display

# Setup Logger
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

## 1. Environment Setup

In [9]:
load_dotenv()

# API Config
os.environ["OPENAI_API_KEY"] = os.getenv("LITELLM_ILSP_EVAL_API_KEY")
os.environ["OPENAI_API_BASE"] = os.getenv("LITELLM_HOST")
api_base = os.getenv("LITELLM_HOST")

# Output Config
results_dir = Path(os.getenv("RESULTS_DIR", "tmp"))
results_dir.mkdir(parents=True, exist_ok=True)

models_to_test = ["gemma3-27b-it", "krikri-dpo-context"]
logger.info(f"Target models: {models_to_test}")

2026-01-07 13:41:52 - INFO - Target models: ['gemma3-27b-it', 'krikri-dpo-context']


## 2. Load and Prepare Dataset

In [10]:
logger.info("Loading dataset...")
original_dataset = load_dataset("PennyK98/protipa_exams_dataset", split="test")

def filter_dataset(dataset):
    filtered = []
    subjects = ['ΓΛΩΣΣΑ', 'ΜΑΘΗΜΑΤΙΚΑ']
    for item in dataset:
        if (item['subject'] in subjects and 
            item['exercise_type'] == 'Multiple Choice' and 
            ',' not in str(item['answer_index'])):
            filtered.append(item)
    return filtered

all_filtered = filter_dataset(original_dataset)
logger.info(f"Filtered dataset size: {len(all_filtered)}")

# Sample for pilot test
pilot_100 = random.sample(all_filtered, min(len(all_filtered), 100))

# Save temporary JSON for lm_eval ingestion
json_path = (results_dir / "pilot_data_test.json").resolve()
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(pilot_100, f, ensure_ascii=False, indent=4)

logger.info(f"Pilot data (100 samples) saved to: {json_path}")

2026-01-07 13:41:52 - INFO - Loading dataset...
2026-01-07 13:41:54 - INFO - Filtered dataset size: 228
2026-01-07 13:41:54 - INFO - Pilot data (100 samples) saved to: /data/home/prokopis/src/protipa_exams_dataset/tmp/pilot_data_test.json


## 3. Define Evaluation Task Template

In [ ]:
task_config = {
    "task": "greek_protipa_exams",
    "dataset_path": "json",
    "dataset_kwargs": {
        "data_files": str(json_path)  # FIXED: Convert Path object to string
    },
    "test_split": "train",
    "output_type": "generate_until",
    "doc_to_text": (
        "{% if input %}{{input}}\n{% endif %}"
        "Ερώτηση: {{question}}\n"
        "Επιλογές:\n"
        "{% for choice in choices %}"
        "{{loop.index0}}. {{choice}}\n"
        "{% endfor %}\n"
        "### ΟΔΗΓΙΑ ΜΟΡΦΟΠΟΙΗΣΗΣ (CRITICAL)\n"
        "Πρέπει να παρέχεις ΜΟΝΟ τον αριθμό του δείκτη (index) της σωστής επιλογής.\n"
        "Μην επεξηγείς και μην γράφεις ολόκληρες προτάσεις.\n\n"
        "❌ ΛΑΘΟΣ: \"Η σωστή επιλογή είναι η 2.\"\n"
        "❌ ΛΑΘΟΣ: \"Ας υποθέσουμε ότι το κλάσμα είναι 1/12, άρα η απάντηση είναι 2.\"\n"
        "❌ ΛΑΘΟΣ: \"(2)\"\n"
        "❌ ΛΑΘΟΣ: \"2.\"\n"
        "✅ ΣΩΣΤΟ: 2\n\n"
        "Απάντηση: "
    ),
    "doc_to_target": "{{ (answer_index | string).split(',')[0] }}",
    "generation_kwargs": {
        "until": ["\n"],
        "max_gen_toks": 50,
        "do_sample": False,
        "temperature": 0.0 
    },
    "filter_list": [
        {
            "name": "strict-match",
            "filter": [
                {"function": "regex", "regex_pattern": r"(?<![0-9/])([0-9])(?![0-9/])"},
                {"function": "take_first"}
            ]
        }
    ],
    "metric_list": [
        {"metric": "exact_match", "aggregation": "mean", "higher_is_better": True}
    ]
}

# Workaround for Task Config loading
task_dir = results_dir
task_dir.mkdir(parents=True, exist_ok=True)
with open(task_dir / "greek_protipa.yaml", "w", encoding='utf-8') as f:
    yaml.dump(task_config, f, allow_unicode=True)

custom_task = ConfigurableTask(config=task_config)
task_dict = {"greek_protipa_exams": custom_task}
logger.info("Evaluation task defined successfully.")

Generating train split: 0 examples [00:00, ? examples/s]

2026-01-07 13:41:54 - INFO - Evaluation task defined successfully.


## 4. Run Evaluation

In [15]:
comparison_results = {}
all_samples = {}

EVAL_LIMIT = 30  # Adjust this to run more/less samples

for model_name in models_to_test:
    logger.info(f"Starting evaluation for model: {model_name}")
    try:
        # Construct endpoint URL
        chat_api_url = api_base
        if not chat_api_url.endswith("/chat/completions"):
            chat_api_url = chat_api_url.rstrip("/") + "/chat/completions"

        model = OpenAIChatCompletion(
            model=model_name,
            base_url=chat_api_url,
            num_fewshot=0,
            eos_string="<|end_of_text|>",
            max_retries=10,
            num_concurrent=1
        )

        results = lm_eval.evaluate(
            lm=model,
            task_dict=task_dict,
            limit=EVAL_LIMIT,
            apply_chat_template=True
        )
        
        # 1. Store Summary Metrics
        scores = results['results']['greek_protipa_exams']
        comparison_results[model_name] = scores
        
        # 2. Collect Individual Samples for the table
        if 'samples' in results and 'greek_protipa_exams' in results['samples']:
            samples = results['samples']['greek_protipa_exams']
            for i, s in enumerate(samples):
                if i not in all_samples:
                    doc = s.get('doc', {})
                    all_samples[i] = {
                        "Question": doc.get('question', 'N/A'),
                        "Subject": doc.get('subject', 'N/A'),
                        "Year": doc.get('year', 'N/A'),
                        "Level": doc.get('school_level', 'N/A'),
                        "Ground Truth": str(doc.get('answer_index', 'N/A')),
                        "Choices": " | ".join(doc.get('choices', [])),
                        "Model Predictions": {},
                        "Raw Responses": {}
                    }
                
                all_samples[i]["Model Predictions"][model_name] = s.get('filtered_resps', ["N/A"])[0]
                all_samples[i]["Raw Responses"][model_name] = s.get('resps', [["N/A"]])[0][0]

        acc = scores.get('exact_match,strict-match', scores.get('acc', 0.0))
        logger.info(f"Success! {model_name} Accuracy: {acc:.2%}")
        
        logger.info("Waiting 2 seconds before next model to avoid rate-limits...")
        time.sleep(2)

    except Exception as e:
        logger.error(f"Error evaluating {model_name}: {e}")
        logger.error(traceback.format_exc())
        time.sleep(2)

2026-01-07 13:46:33 - INFO - Starting evaluation for model: gemma3-27b-it
2026-01-07 13:46:33 - INFO - Using max length 2048 - 1
2026-01-07 13:46:33 - INFO - Concurrent requests are disabled. To enable concurrent requests, set `num_concurrent` > 1.
2026-01-07 13:46:33 - INFO - Using tokenizer None
2026-01-07 13:46:33 - WARNING - Chat template formatting change affects loglikelihood and multiple-choice tasks. See docs/chat-template-readme.md for details.
2026-01-07 13:46:33 - INFO - Building contexts for greek_protipa_exams on rank 0...
100%|██████████| 30/30 [00:00<00:00, 418.14it/s]
2026-01-07 13:46:33 - INFO - Running generate_until requests
2026-01-07 13:46:33 - INFO - Tokenized requests are disabled. Context + generation length is not checked.
Requesting API: 100%|██████████| 30/30 [00:13<00:00,  2.17it/s]
2026-01-07 13:46:47 - INFO - Success! gemma3-27b-it Accuracy: 36.67%
2026-01-07 13:46:47 - INFO - Waiting 2 seconds before next model to avoid rate-limits...
2026-01-07 13:46:49 

## 5. Results Table

In [19]:
if all_samples:
    table_data = []
    for idx, data in all_samples.items():
        row = {
            "ID": idx,
            "Subject": data["Subject"],
            "Year": data["Year"],
            "Level": data["Level"],
            "Question": data["Question"],
            "Choices": data["Choices"],
            "Ground Truth": data["Ground Truth"]
        }
        for m in models_to_test:
            # Add both the extracted prediction AND the raw text from the model
            row[f"{m}_pred"] = data["Model Predictions"].get(m, "N/A")
            # row[f"{m}_raw"] = data["Raw Responses"].get(m, "N/A") 
            
        table_data.append(row)
    
    df_results = pd.DataFrame(table_data)
    
    # Save CSV
    results_file = results_dir / "eval_results_table.csv"
    df_results.to_csv(results_file, index=False, encoding='utf-8')
    logger.info(f"Table saved to: {results_file}")
    
    # This will now display the raw responses in the table below
    display(df_results.head(10)) 

2026-01-07 13:50:00 - INFO - Table saved to: tmp/eval_results_table.csv


,ID,Subject,Year,Level,Question,Choices,Ground Truth,gemma3-27b-it_pred,krikri-dpo-context_pred
0,0,ΓΛΩΣΣΑ,2025,ΓΥΜΝΑΣΙΟ,Ποια είδη προτάσεων αναγνωρίζετε στην περίοδο ...,"Α. κύρια, τελική, τελική, τελική, τελική | Β. ...",0,1,2
1,1,ΓΛΩΣΣΑ,2020,ΓΥΜΝΑΣΙΟ,δεσπόζει: ποια από τις παρακάτω λέξεις είναι σ...,Α. εξουσιάζει | Β. κυριαρχεί | Γ. προβάλλει | ...,1,1,2
2,2,ΜΑΘΗΜΑΤΙΚΑ,2024,ΛΥΚΕΙΟ,Ποιο από τα παρακάτω ισχύει για τους αριθμούς ...,Α. είναι αντίστροφοι | Β. είναι αντίθετοι | Γ....,0,0,2
3,3,ΜΑΘΗΜΑΤΙΚΑ,2025,ΛΥΚΕΙΟ,Ποιο είναι το ανάπτυγμα του γινομένου $(x-1)(x...,A. $x^{8}+1$ | B. $x^{8}-1$ | Γ. $x^{8}+2x^{4}...,1,0,2
4,4,ΜΑΘΗΜΑΤΙΚΑ,2016,ΛΥΚΕΙΟ,Οι λύσεις τις εξίσωσης $\frac{x^{2}-x}{1-x}=1$...,A. 1 | B. 2 | Γ. +1 | Δ. -1 | E. 0,3,0,2
5,5,ΓΛΩΣΣΑ,2025,ΓΥΜΝΑΣΙΟ,Στην περίοδο «κανείς δε μας έστειλε να κοιμηθο...,Α. μονόπτωτο | Β. δίπτωτο | Γ. αμετάβατο | Δ. ...,0,1,2
6,6,ΜΑΘΗΜΑΤΙΚΑ,2016,ΓΥΜΝΑΣΙΟ,Πόσα μηδενικά έχει το αποτέλεσμα της πράξης: $...,Α. 6 | Β. 7 | Γ. 10 | Δ. 12 | Ε. 14,4,3,2
7,7,ΜΑΘΗΜΑΤΙΚΑ,2020,ΛΥΚΕΙΟ,Στο παρακάτω σχήμα φαίνονται δύο μεζούρες μέτρ...,"A. 5,3 | B. 7 | Г. 6,5 | Δ. 6,75",3,3,2
8,8,ΓΛΩΣΣΑ,2020,ΛΥΚΕΙΟ,"Ειδικά οι μαθητές θα πρέπει, με την **αρωγή** ...",α. παρότρυνση | β. παρακίνηση | γ. βοήθεια | δ...,2,2,2
9,9,ΜΑΘΗΜΑΤΙΚΑ,2022,ΛΥΚΕΙΟ,Για την ισότητα $1636=43\cdot37+45$ είναι αλήθ...,Α. αντιστοιχεί σε Ευκλείδεια Διαίρεση με διαιρ...,3,0,2


## 6. Performance Summary

In [20]:
if comparison_results:
    summary_data = []
    for model, metrics in comparison_results.items():
        acc = metrics.get('exact_match,strict-match', metrics.get('acc', 0.0))
        summary_data.append({"Model": model, "Accuracy": acc})
    
    df_summary = pd.DataFrame(summary_data)
    display(df_summary)

,Model,Accuracy
0,gemma3-27b-it,0.366667
1,krikri-dpo-context,0.233333
